# Autoencoder learning-rate sweep

Same run as `autoencoder_test.py`. Train a new Fizeau PSF autoencoder at each learning rate, write every held-out frame and its reconstruction to FITS, and compare the runs with two plots: training loss, and reconstructions at low, median, and high OPD.

In [ ]:
import sys

# let the machine find the packages
sys.path.insert(0, "/home/ipa/quanz/user_accounts/espalding/dependencies")

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from astropy.io import fits
from torch import optim
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

from modules_autoencoder import FizeauDataset, autoenc

## Setup

Eighty of the 100 frames are used for training. The other twenty are held out. Each learning rate gets its own freshly initialized network.

In [ ]:
training_frac = 0.8
N = 100
n_training = int(N * training_frac)
learning_rates = np.array([0.01, 0.001, 0.0001])
batch_size = 20
training_epochs = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# where is the input data?
simulated_images_varied_OPD = Path("/home/ipa/quanz/user_accounts/espalding/fake_datasets/opd_only/")
# where should the reconstructed images go?
out_root = Path("/home/ipa/quanz/user_accounts/espalding/reconstructions")

for directory in (simulated_images_varied_OPD, out_root):
    if not directory.is_dir():
        directory.mkdir(parents=True)

## Data

`FizeauDataset` log-scales each PSF and stretches it to [0, 1]. That is the image the network sees, and the image the plots below show. The split is a fresh shuffle every time this cell runs.

In [ ]:
files = sorted(simulated_images_varied_OPD.glob("*.fits"))
dataset = FizeauDataset(files, log=True)

idxs = np.arange(0, N)
np.random.shuffle(idxs)
train_idxs = idxs[:n_training]
test_idxs = idxs[n_training:]

train_dataset = Subset(dataset, train_idxs)
test_dataset = Subset(dataset, test_idxs)

# batch size: number of images to pass through the network at once; these get shuffled if shuffle=True
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
# batch_size=1 makes each test image its own tensor, with the batch dimension the loader adds
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"{len(dataset)} frames, {len(train_dataset)} train, {len(test_dataset)} test")

## Training

Loss is the mean absolute error between the reconstruction and the input frame. The model has to sit on the same device as the batch.

In [ ]:
loss = nn.L1Loss()

loss_lr = np.zeros((len(learning_rates), len(train_loader) * training_epochs))
models = []

for i, learning_rate in enumerate(learning_rates):
    time_0 = time.time()
    print(f"learning rate: {learning_rate}")

    model = autoenc(1).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
    loss_per_epoch = []

    for epoch in range(training_epochs):
        print(f"Epoch [{epoch + 1}/{training_epochs}]")
        for batch_index, (batch, _) in enumerate(tqdm(train_loader)):
            batch = batch.to(device)

            _, reconstructed, _ = model(batch)
            loss_ = loss(reconstructed, batch)
            loss_per_epoch.append(loss_.detach().cpu().item())

            optimizer.zero_grad()
            loss_.backward()
            optimizer.step()

    loss_lr[i, :] = np.array(loss_per_epoch)
    models.append(model)

    time_diff_train = time.time() - time_0
    print(f"Elapsed time to train with learning rate {learning_rate:.0e}: {time_diff_train:.2f} seconds")

## Training loss

Each point is the mean L1 over the batches in that epoch. A lower curve reconstructed the training frames more closely. The three curves are separate networks, so a gap here is the learning rate.

In [ ]:
steps_per_epoch = loss_lr.shape[1] // training_epochs
epoch_loss = loss_lr.reshape(len(learning_rates), training_epochs, steps_per_epoch).mean(axis=2)
epoch_axis = np.arange(1, training_epochs + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, learning_rate in enumerate(learning_rates):
    ax.plot(epoch_axis, epoch_loss[i], label=f"lr = {learning_rate:g}")
    print(f"lr {learning_rate:g}: final epoch mean L1 = {epoch_loss[i, -1]:.4f}")

ax.set_xlabel("epoch")
ax.set_ylabel("mean L1 loss")
ax.set_title("Training loss by learning rate")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Test reconstructions

Every held-out frame is written as a pair of FITS files, original and reconstructed, under `reconstructions/lr_<rate>/`. A dataset item is `(channel, height, width)`; the network is given a leading batch dimension so this forward pass matches training.

The figure uses three held-out frames: lowest OPD, median OPD, and highest OPD. The number under each reconstruction is its mean absolute error against the original, on the same [0, 1] scale as the loss.

In [ ]:
subset = test_loader.dataset

for learning_rate, model in zip(learning_rates, models):
    time_0 = time.time()
    model.eval()
    lr_dir = out_root / f"lr_{learning_rate:.0e}"
    if not lr_dir.is_dir():
        lr_dir.mkdir(parents=True)

    with torch.no_grad():
        for i, test_data in enumerate(subset):
            test_img, opd = test_data
            batch = test_img.unsqueeze(0).to(device)
            _, reconstructed, _ = model(batch)

            src = Path(subset.dataset.files[subset.indices[i]])
            hdr = fits.Header()
            hdr["OPD"] = float(opd)
            hdr["LR"] = float(learning_rate)
            hdr["SOURCE"] = src.name

            fits.writeto(
                lr_dir / f"{src.stem}_original.fits",
                test_img.numpy()[0],
                hdr,
                overwrite=True,
            )
            fits.writeto(
                lr_dir / f"{src.stem}_reconstructed.fits",
                reconstructed.detach().cpu().numpy()[0, 0],
                hdr,
                overwrite=True,
            )

    print(f"wrote {len(subset)} frame pairs to {lr_dir}")
    time_diff_test = time.time() - time_0
    print(f"Elapsed time to test with learning rate {learning_rate:.0e}: {time_diff_test:.2f} seconds")

In [ ]:
opd_values = np.array([float(subset[i][1]) for i in range(len(subset))])
order = np.argsort(opd_values)
show_idxs = [int(order[0]), int(order[len(order) // 2]), int(order[-1])]

fig, axes = plt.subplots(
    len(show_idxs),
    1 + len(learning_rates),
    figsize=(2.6 * (1 + len(learning_rates)), 2.7 * len(show_idxs)),
    squeeze=False,
)

for row, idx in enumerate(show_idxs):
    test_img, opd = subset[idx]
    original = test_img.numpy()[0]
    axes[row, 0].imshow(original, vmin=0, vmax=1, cmap="magma")
    axes[row, 0].set_ylabel(f"OPD\n{float(opd) * 1e6:.1f} µm")
    if row == 0:
        axes[row, 0].set_title("original")

    batch = test_img.unsqueeze(0).to(device)
    for col, (learning_rate, model) in enumerate(zip(learning_rates, models), start=1):
        model.eval()
        with torch.no_grad():
            _, reconstructed, _ = model(batch)
        recon = reconstructed.detach().cpu().numpy()[0, 0]
        mae = float(np.mean(np.abs(recon - original)))
        axes[row, col].imshow(recon, vmin=0, vmax=1, cmap="magma")
        axes[row, col].set_xlabel(f"L1 = {mae:.3f}")
        if row == 0:
            axes[row, col].set_title(f"lr = {learning_rate:g}")

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Held-out frames: original and reconstruction", y=1.01)
fig.tight_layout()
plt.show()